# 🚗 Car Dataset — Real-time Pipeline
Voer de cellen van boven naar beneden uit.

## 1 · Imports & Configuratie

In [1]:
import pandas as pd
import logging
import os
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
import glob


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)


INPUT_FOLDER = './dataset'
OUTPUT_PATH  = './output/cars_processed.csv'
BLOB_NAME    = 'cars/cars_processed.csv'

## 2 · Reader

In [2]:
# Zoek alle bestanden met de extensie .csv in de opgegeven INPUT_FOLDER
files = glob.glob(os.path.join(INPUT_FOLDER, '*.csv'))

# Controleer of er geen CSV-bestanden gevonden zijn in de map
if not files:
    # Log een foutmelding met het pad van de map waar gezocht werd
    logger.error(f"Geen CSV gevonden in {INPUT_FOLDER}")
else:
    # Neem het eerste gevonden CSV-bestand uit de lijst
    file_path = files[0]
    # Probeer het bestand in te lezen, vang eventuele fouten op
    try:
        # Lees het CSV-bestand in als een pandas DataFrame
        df = pd.read_csv(file_path)
        # Log een succesbericht met het bestandspad en de dimensies van het DataFrame
        logger.info(f"Gelezen: {file_path} — {df.shape[0]:,} rijen x {df.shape[1]} kolommen")
    except Exception as e:
        # Log een foutmelding als het inlezen mislukt, inclusief de foutdetails
        logger.error(f"Fout bij inlezen {file_path}: {e}")

2026-05-03 21:35:16,798 - INFO - Gelezen: ./dataset/cars.csv — 160 rijen x 13 kolommen


## 3 · Validator
Elke check logt het **aantal** slechte rijen, de **rij-indices** én de **waarden** die afgekeurd werden.

In [3]:
# Maak de 'logs' map aan als die nog niet bestaat, zonder fout als die al bestaat
os.makedirs('./logs', exist_ok=True)
# Stel het pad in voor het logbestand
log_path = './logs/cars_validation.log'


# Definieer een hulpfunctie om berichten toe te voegen aan het logbestand
def log(msg):
    # Open het logbestand in append-modus met UTF-8 encoding
    with open(log_path, 'a', encoding='UTF-8') as f:
        # Schrijf het bericht gevolgd door een nieuwe regel
        f.write(msg + '\n')


# Log een visuele scheidingslijn om een nieuwe run te markeren
log('=' * 60)
# Log het tijdstip van de huidige validatierun
log(f"Validation run — {pd.Timestamp.now()}")
# Log het aantal rijen en kolommen vóór validatie
log(f"Rows before: {len(df)}, Columns: {len(df.columns)}")
# Log een afsluitende scheidingslijn
log('=' * 60)


# ── Duplicates ────────────────────────────────────────────────────────────
# Detecteer dubbele rijen in het volledige DataFrame
dup_mask = df.duplicated()
# Controleer of er minstens één duplicaat aanwezig is
if dup_mask.any():
    # Sla de dubbele rijen op in een aparte variabele voor logging
    dup_rows = df[dup_mask]
    # Log het aantal verwijderde duplicaatrijen
    log(f"[DROP] duplicates: {dup_mask.sum()} dubbele rijen")
    # Log de DataFrame-indices van de duplicaatrijen
    log(f"       Indices: {dup_rows.index.tolist()}")
    # Log de car_id's van de verwijderde duplicaten
    log(f"       car_ids: {dup_rows['car_id'].tolist()}")
    # Verwijder de duplicaten en reset de index
    df = df[~dup_mask].reset_index(drop=True)


# ── Mandatory null checks ─────────────────────────────────────────────────
# Itereer over alle verplichte kolommen en controleer op null-waarden
for col in ['car_id', 'brand', 'year', 'mileage_km', 'fuel_type', 'price_eur']:
    # Maak een masker van rijen waar de kolom een null-waarde bevat
    mask = df[col].isna()
    # Controleer of er nulls aanwezig zijn in deze kolom
    if mask.any():
        # Sla de rijen met null-waarden op voor logging
        bad = df[mask]
        # Log het aantal null-waarden in de kolom
        log(f"[DROP] {col}: {mask.sum()} null values")
        # Log de indices van de betrokken rijen
        log(f"       Indices: {bad.index.tolist()}")
        # Log de car_id's van de verwijderde rijen
        log(f"       car_ids: {bad['car_id'].tolist()}")
        # Verwijder rijen met null-waarden en reset de index
        df = df[~mask].reset_index(drop=True)


# ── brand: valid brands ───────────────────────────────────────────────────
# Definieer de lijst van toegestane merkwaarden
valid_brands = ['Toyota','Ford','BMW','Mercedes','Volkswagen','Audi','Honda','Hyundai',
                'Renault','Peugeot','Skoda','Seat','Nissan','Kia','Mazda','Volvo',
                'Citroën','Dacia','Opel','Fiat']
# Maak een masker van rijen waarvan het merk niet in de geldige lijst staat
mask = ~df['brand'].isin(valid_brands)
# Controleer of er ongeldige merkwaarden aanwezig zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een ongeldig merk
    log(f"[DROP] brand: {mask.sum()} invalid values")
    # Log de indices van de betrokken rijen
    log(f"       Indices: {bad.index.tolist()}")
    # Log de ongeldige merkwaarden zelf
    log(f"       Values: {bad['brand'].tolist()}")
    # Log de car_id's van de verwijderde rijen
    log(f"       car_ids: {bad['car_id'].tolist()}")
    # Verwijder rijen met ongeldige merken en reset de index
    df = df[~mask].reset_index(drop=True)


# ── year: 1990–2025 ───────────────────────────────────────────────────────
# Maak een masker van rijen waarvan het bouwjaar buiten het bereik 1990–2025 valt
mask = (df['year'] < 1990) | (df['year'] > 2025)
# Controleer of er jaren buiten het geldige bereik zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een ongeldig bouwjaar
    log(f"[DROP] year: {mask.sum()} out of range (1990-2025)")
    # Log de indices van de betrokken rijen
    log(f"       Indices: {bad.index.tolist()}")
    # Log de ongeldige jaarwaarden
    log(f"       Values: {bad['year'].tolist()}")
    # Log de car_id's van de verwijderde rijen
    log(f"       car_ids: {bad['car_id'].tolist()}")
    # Verwijder rijen met een ongeldig bouwjaar en reset de index
    df = df[~mask].reset_index(drop=True)


# ── mileage_km: >= 0 ──────────────────────────────────────────────────────
# Maak een masker van rijen met een negatieve kilometerstand
mask = df['mileage_km'] < 0
# Controleer of er negatieve kilometerstandwaarden aanwezig zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een negatieve kilometerstand
    log(f"[DROP] mileage_km: {mask.sum()} negative values")
    # Log de indices van de betrokken rijen
    log(f"       Indices: {bad.index.tolist()}")
    # Log de negatieve kilometerstandwaarden
    log(f"       Values: {bad['mileage_km'].tolist()}")
    # Log de car_id's van de verwijderde rijen
    log(f"       car_ids: {bad['car_id'].tolist()}")
    # Verwijder rijen met negatieve kilometerstand en reset de index
    df = df[~mask].reset_index(drop=True)


# ── fuel_type: valid values ───────────────────────────────────────────────
# Definieer de lijst van toegestane brandstoftypes
valid_fuels = ['Petrol', 'Diesel', 'Electric', 'Hybrid']
# Maak een masker van rijen waarvan het brandstoftype niet geldig is
mask = ~df['fuel_type'].isin(valid_fuels)
# Controleer of er ongeldige brandstoftypes aanwezig zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een ongeldig brandstoftype
    log(f"[DROP] fuel_type: {mask.sum()} invalid values")
    # Log de indices van de betrokken rijen
    log(f"       Indices: {bad.index.tolist()}")
    # Log de ongeldige brandstoftypewaarden
    log(f"       Values: {bad['fuel_type'].tolist()}")
    # Log de car_id's van de verwijderde rijen
    log(f"       car_ids: {bad['car_id'].tolist()}")
    # Verwijder rijen met ongeldige brandstoftypes en reset de index
    df = df[~mask].reset_index(drop=True)


# ── price_eur: > 0 ────────────────────────────────────────────────────────
# Maak een masker van rijen met een prijs van 0 of lager
mask = df['price_eur'] <= 0
# Controleer of er niet-positieve prijswaarden aanwezig zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een ongeldige prijs
    log(f"[DROP] price_eur: {mask.sum()} non-positive values")
    # Log de indices van de betrokken rijen
    log(f"       Indices: {bad.index.tolist()}")
    # Log de ongeldige prijswaarden
    log(f"       Values: {bad['price_eur'].tolist()}")
    # Log de car_id's van de verwijderde rijen
    log(f"       car_ids: {bad['car_id'].tolist()}")
    # Verwijder rijen met niet-positieve prijzen en reset de index
    df = df[~mask].reset_index(drop=True)


# ── engine_cc: null → DROP, <= 0 → DROP ──────────────────────────────────
# Maak een masker van rijen met null of een niet-positief cilindervolume
mask = df['engine_cc'].isna() | (df['engine_cc'] <= 0)
# Controleer of er ongeldige motorinhoudwaarden aanwezig zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een ongeldige motorinhoud
    log(f"[DROP] engine_cc: {mask.sum()} null or non-positive values")
    # Log de indices van de betrokken rijen
    log(f"       Indices: {bad.index.tolist()}")
    # Log de car_id's van de verwijderde rijen
    log(f"       car_ids: {bad['car_id'].tolist()}")
    # Verwijder rijen met ongeldige motorinhoud en reset de index
    df = df[~mask].reset_index(drop=True)


# ── horsepower: null → DROP, <= 0 → DROP ─────────────────────────────────
# Maak een masker van rijen met null of een niet-positief vermogen
mask = df['horsepower'].isna() | (df['horsepower'] <= 0)
# Controleer of er ongeldige vermogenswaarden aanwezig zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een ongeldig vermogen
    log(f"[DROP] horsepower: {mask.sum()} null or non-positive values")
    # Log de indices van de betrokken rijen
    log(f"       Indices: {bad.index.tolist()}")
    # Log de car_id's van de verwijderde rijen
    log(f"       car_ids: {bad['car_id'].tolist()}")
    # Verwijder rijen met ongeldig vermogen en reset de index
    df = df[~mask].reset_index(drop=True)


# ── num_doors: null → DROP, invalid → DROP ───────────────────────────────
# Maak een masker van rijen met null of een ongeldig aantal deuren (niet 2, 3, 4 of 5)
mask = df['num_doors'].isna() | ~df['num_doors'].isin([2, 3, 4, 5])
# Controleer of er ongeldige deurenaantallen aanwezig zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een ongeldig aantal deuren
    log(f"[DROP] num_doors: {mask.sum()} null or invalid values (expected 2-5)")
    # Log de indices van de betrokken rijen
    log(f"       Indices: {bad.index.tolist()}")
    # Log de car_id's van de verwijderde rijen
    log(f"       car_ids: {bad['car_id'].tolist()}")
    # Verwijder rijen met ongeldig aantal deuren en reset de index
    df = df[~mask].reset_index(drop=True)


# ── co2_g_per_km: Electric null → 0, others null → DROP, < 0 → DROP ──────
# Maak een masker van alle rijen met een null CO2-waarde
mask_null = df['co2_g_per_km'].isna()
# Controleer of er null CO2-waarden aanwezig zijn
if mask_null.any():
    # Identificeer elektrische voertuigen met een null CO2-waarde (logisch → imputeer 0)
    electric_null = mask_null & (df['fuel_type'] == 'Electric')
    # Identificeer niet-elektrische voertuigen met een null CO2-waarde (ongeldig → DROP)
    other_null = mask_null & (df['fuel_type'] != 'Electric')
    # Verwerk de null CO2-waarden voor elektrische voertuigen
    if electric_null.any():
        # Log het aantal geïmputeerde CO2-waarden voor elektrische voertuigen
        log(f"[FLAG] co2_g_per_km: {electric_null.sum()} Electric null values — imputed 0")
        # Log de indices van de geïmputeerde rijen
        log(f"       Indices: {df[electric_null].index.tolist()}")
        # Log de car_id's van de geïmputeerde rijen
        log(f"       car_ids: {df[electric_null]['car_id'].tolist()}")
        # Stel de CO2-waarde in op 0 voor elektrische voertuigen zonder CO2-waarde
        df.loc[electric_null, 'co2_g_per_km'] = 0
    # Verwerk de null CO2-waarden voor niet-elektrische voertuigen
    if other_null.any():
        # Sla de ongeldige rijen op voor logging
        bad = df[other_null]
        # Log het aantal verwijderde rijen wegens ontbrekende CO2-waarde
        log(f"[DROP] co2_g_per_km: {other_null.sum()} null values (non-Electric) — DROPPED")
        # Log de indices van de verwijderde rijen
        log(f"       Indices: {bad.index.tolist()}")
        # Log de car_id's van de verwijderde rijen
        log(f"       car_ids: {bad['car_id'].tolist()}")
        # Verwijder rijen met null CO2-waarde (niet-elektrisch) en reset de index
        df = df[~other_null].reset_index(drop=True)

# Maak een masker van rijen met een negatieve CO2-waarde
mask = df['co2_g_per_km'] < 0
# Controleer of er negatieve CO2-waarden aanwezig zijn
if mask.any():
    # Sla de ongeldige rijen op voor logging
    bad = df[mask]
    # Log het aantal rijen met een negatieve CO2-waarde inclusief de waarden zelf
    log(f"[DROP] co2_g_per_km: {mask.sum()} negative values")
    log(f"       Indices: {bad.index.tolist()}, Values: {bad['co2_g_per_km'].tolist()}")
    # Verwijder rijen met negatieve CO2-waarden en reset de index
    df = df[~mask].reset_index(drop=True)


# Log het aantal overgebleven rijen na alle validatiestappen
log(f"Rows after: {len(df)}")
# Log een afsluitend bericht om het einde van de validatierun aan te duiden
log("Validation complete.\n")
# Print een samenvatting naar de console met het aantal resterende rijen
print(f"Done. Rows remaining: {len(df)}. Check logs/cars_validation.log")

Done. Rows remaining: 32. Check logs/cars_validation.log


## 4 · Processor

In [4]:
# Haal het huidige kalenderjaar op via pandas Timestamp
current_year = pd.Timestamp.now().year

# Bereken de leeftijd van elk voertuig in jaren door het bouwjaar af te trekken van het huidige jaar
df['car_age_years'] = current_year - df['year'].astype(int)


# Deel de prijs op in 4 categorieën: Budget, Mid-range, Premium en Luxury
df['price_category'] = pd.cut(
    # Kolom waarop de indeling wordt toegepast
    df['price_eur'],
    # Grenzen van de prijsklassen (0–10k, 10k–30k, 30k–60k, 60k+)
    bins=[0, 10000, 30000, 60000, float('inf')],
    # Labels die aan elke klasse worden toegekend
    labels=['Budget', 'Mid-range', 'Premium', 'Luxury'],
    # right=False betekent dat de linkergrens wél inbegrepen is, de rechtergrens niet
    right=False
)


# Deel de kilometerstand op in 3 categorieën: Low, Medium en High
df['mileage_category'] = pd.cut(
    # Kolom waarop de indeling wordt toegepast
    df['mileage_km'],
    # Grenzen van de kilometerklassen (0–50k, 50k–150k, 150k+)
    bins=[0, 50000, 150000, float('inf')],
    # Labels die aan elke klasse worden toegekend
    labels=['Low', 'Medium', 'High'],
    # right=False: linkergrens inbegrepen, rechtergrens niet
    right=False,
    # include_lowest=True zorgt dat waarde 0 ook in de eerste bin valt
    include_lowest=True
)


# Bereken het vermogen per 100cc als maatstaf voor motorefficiëntie, afgerond op 2 decimalen
df['hp_per_100cc'] = (df['horsepower'] / df['engine_cc'] * 100).round(2)


# Maak een binaire kolom: 1 als het voertuig elektrisch is, anders 0
df['is_electric'] = (df['fuel_type'] == 'Electric').astype(int)


# Log een informatief bericht met de uiteindelijke dimensies van het verwerkte DataFrame
logger.info(f"Processing compleet: {df.shape[0]:,} rijen x {df.shape[1]} kolommen.")
# Toon de eerste 5 rijen van het DataFrame als snelle visuele controle
df.head()

2026-05-03 21:35:16,829 - INFO - Processing compleet: 32 rijen x 18 kolommen.


,car_id,brand,model,year,mileage_km,fuel_type,engine_cc,horsepower,price_eur,color,transmission,num_doors,co2_g_per_km,car_age_years,price_category,mileage_category,hp_per_100cc,is_electric
0,7,BMW,Model_7,2004.0,100000.0,Hybrid,4400.0,330.0,25000,Red,Manual,4.0,195.0,22,Mid-range,Medium,7.50,0
1,9,Hyundai,Model_9,2018.0,95000.0,Hybrid,2000.0,180.0,29500,White,NaN,4.0,0.0,8,Mid-range,Medium,9.00,0
2,12,Hyundai,Model_12,2023.0,30000.0,Hybrid,4200.0,110.0,33000,Red,Automatic,5.0,210.0,3,Premium,Low,2.62,0
3,15,Audi,Model_15,2021.0,160000.0,Hybrid,2400.0,300.0,42000,Silver,Automatic,2.0,120.0,5,Premium,High,12.50,0
4,18,Hyundai,Model_18,2019.0,290000.0,Hybrid,2900.0,350.0,2000,Silver,NaN,5.0,10.0,7,Budget,High,12.07,0


## 5 · Writer

In [5]:
# Laad de omgevingsvariabelen uit het .env-bestand in de huidige omgeving
load_dotenv()

# Stel het lokale uitvoerpad in voor het CSV-bestand
OUTPUT_PATH = 'output/cars.csv'

# Maak de 'output' map aan als die nog niet bestaat, zonder fout als die al bestaat
os.makedirs('output', exist_ok=True)
# Sla het DataFrame op als CSV-bestand zonder de pandas-index mee te schrijven
df.to_csv(OUTPUT_PATH, index=False)

# Maak een verbinding met Azure Blob Storage via de connection string uit de omgevingsvariabelen
client = BlobServiceClient.from_connection_string(os.getenv('AZURE_STORAGE_CONNECTION_STRING'))
# Haal een referentie op naar het specifieke blob-bestand binnen de opgegeven container
blob   = client.get_blob_client(container="yellow-taxi-data", blob="cars.csv")

# Open het lokaal opgeslagen CSV-bestand in binaire leesmodus
with open(OUTPUT_PATH, 'rb') as f:
    # Upload de bestandsinhoud naar Azure Blob Storage en overschrijf een eventueel bestaande blob
    blob.upload_blob(f, overwrite=True)

# Bevestig in de console dat het bestand zowel lokaal als op Azure is opgeslagen
print("Klaar. Opgeslagen lokaal en naar Azure.")

2026-05-03 21:35:16,846 - INFO - Request URL: 'https://yellowtaxinachatkaran.blob.core.windows.net/yellow-taxi-data/cars.csv'
Request method: 'PUT'
Request headers:
    'Content-Length': '3510'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.19.1 Python/3.12.2 (Linux-6.12.54-linuxkit-aarch64-with-glibc2.36)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'fa02f7e4-4737-11f1-9305-8afa394f6bae'
    'Authorization': 'REDACTED'
A body is sent with the request


2026-05-03 21:35:17,022 - INFO - Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Sun, 03 May 2026 21:35:17 GMT'
    'ETag': '"0x8DEA95BDE514508"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'cc3f59a9-401e-0067-6f44-db3363000000'
    'x-ms-client-request-id': 'fa02f7e4-4737-11f1-9305-8afa394f6bae'
    'x-ms-version': 'REDACTED'
    'x-ms-content-crc64': 'REDACTED'
    'x-ms-request-server-encrypted': 'REDACTED'
    'Date': 'Sun, 03 May 2026 21:35:16 GMT'


Klaar. Opgeslagen lokaal en naar Azure.
